# Frozen prospective localization claims: the genes hyperLOPIT did not place


**What this is.** For every gene with no hyperLOPIT assignment, the niche the data say it is most likely in, with a probability -- written down, hashed and dated now, so an independent localisation source can grade it later without anyone having been able to adjust it (instruction 63).

**How it was made.** One model per niche (boosted trees over every allowed evidence family), fitted on the assigned genes with **every column of the place category held out**, so no localisation measurement can restate the answer. The deployment fit; no frozen partition is touched.

**What it is not.** A fill. No localization question clears the 0.90 bar, so these are predictions to check, never facts in the map.

Frozen 2026-11-08.

In [1]:
import hashlib, json, os, time
import numpy as np, pandas as pd
from parascape import harness as H, datasets as ds, organisms as O
from parascape.question_registry import Question
nodes = pd.read_parquet(os.path.join(REPO, 'parascape/data/nodes.parquet')).set_index('gene_id')
assigned = nodes['lopit_unified'].notna()
niches = nodes.loc[assigned, 'lopit_unified'].value_counts()
def family_of(column):
    found = ds.provenance(column, O.TOXOPLASMA)
    return found.key if found is not None else 'unregistered'
numeric = [c for c in nodes.columns if pd.api.types.is_numeric_dtype(nodes[c]) or pd.api.types.is_bool_dtype(nodes[c])]
families = pd.Series({c: family_of(c) for c in numeric})
print(int(assigned.sum()), 'assigned genes;', int((~assigned).sum()), 'unassigned;', len(niches), 'niches')
niches.to_frame('assigned genes')

3827 assigned genes; 4313 unassigned; 11 niches


,assigned genes
lopit_unified,
nucleus,1262
cytosol,599
mitochondrion,467
apical_secretory,448
pellicle_PM_IMC,324
ER,265
golgi_secretory_vesicular,187
apicoplast,167
ribosome_RNP,68


## 1 · One model per niche, place held out

A niche with fewer than 50 assigned genes is not modelled and is named. The held-out record of this question family is in `question_registry` (frozen-test AUC 0.846-0.933 per niche, with every place column held out); this notebook is the deployment fit on all assigned genes.

In [2]:
MIN = 50
unassigned = nodes.index[~assigned]
strategy = H.Strategy.of('boosted trees (reference)', 'hist_gbt', max_iter=150)
columns, skipped, scores = None, [], {}
for niche, n in niches.items():
    if n < MIN:
        skipped.append(f'{niche} ({n})')
        continue
    q = Question(id=f'prospective_localization_{niche}', text=f'Is this protein in the {niche}?',
                 answer_type='boolean', question_class='B', label_source='nodes.parquet:lopit_unified',
                 label_family=family_of('lopit_unified'), evidence_forbidden=(family_of('lopit_unified'),),
                 label_tier='observed-curated', label_category='place', status='scored')
    features = H.allowed_features(q, nodes[numeric].astype('float64'), families)
    labels = (nodes.loc[assigned, 'lopit_unified'] == niche)
    usable = [c for c in features.columns if features.loc[labels.index, c].notna().any()]
    model = strategy.build().fit(features.loc[labels.index, usable], labels.astype(int))
    scores[niche] = model.predict_proba(features.loc[unassigned, usable])[:, 1]
    columns = usable
print(len(scores), 'niches modelled;', features.shape[1], 'allowed columns of', len(numeric))
print('too few assigned genes to model:', ', '.join(skipped) if skipped else 'none')
P = pd.DataFrame(scores, index=unassigned)
P.round(3).head()

9 niches modelled; 373 allowed columns of 411
too few assigned genes to model: proteasome (31), cytoskeleton (9)


,nucleus,cytosol,mitochondrion,apical_secretory,pellicle_PM_IMC,ER,golgi_secretory_vesicular,apicoplast,ribosome_RNP
gene_id,,,,,,,,,
TGME49_313630,0.632,0.007,0.026,0.018,0.02,0.054,0.009,0.29,0
TGME49_209000,0.678,0.022,0.004,0.106,0.068,0.018,0.004,0.001,0
TGME49_225745,0.723,0.032,0.007,0.045,0.019,0.012,0.007,0.033,0
TGME49_500189,0.588,0.031,0.085,0.012,0.012,0.015,0.007,0.028,0
TGME49_294820,0.054,0.046,0.085,0.068,0.109,0.016,0.009,0.009,0


## 2 · The claim for each unplaced gene

The niche with the highest probability, and how far it is ahead of the next one -- the margin is the part to read: a gene whose best two niches are a hair apart is not placed by this evidence.

In [3]:
order = np.argsort(-P.to_numpy(), axis=1)
best = P.columns.to_numpy()[order[:, 0]]
top = np.take_along_axis(P.to_numpy(), order[:, :2], axis=1)
claims = pd.DataFrame({'gene_id': unassigned.astype(str), 'niche': best,
                       'probability': np.round(top[:, 0], 4), 'margin': np.round(top[:, 0] - top[:, 1], 4),
                       'second_niche': P.columns.to_numpy()[order[:, 1]]})
claims['product'] = nodes.loc[claims.gene_id, 'product'].values if 'product' in nodes else ''
claims = claims.sort_values('probability', ascending=False).reset_index(drop=True)
claims['rank'] = np.arange(1, len(claims) + 1)
print(len(claims), 'claims;', int((claims.margin >= 0.2).sum()), 'with a clear best niche (margin >= 0.2)')
claims.niche.value_counts().to_frame('claims')

4313 claims; 1866 with a clear best niche (margin >= 0.2)


,claims
niche,
mitochondrion,1371
nucleus,1340
pellicle_PM_IMC,772
apical_secretory,386
apicoplast,251
ER,96
golgi_secretory_vesicular,60
cytosol,36
ribosome_RNP,1


## 3 · The freeze

The claims file, its SHA-256, the strategy and evidence hashes, the version and the date go into `manifest.json`. Grading later means: recompute the hash, confirm it matches, then score the claims against the new source -- accuracy on the genes it places, against the commonest niche as chance, and accuracy among the claims with a margin of at least 0.2.

In [4]:
from parascape import __version__
path = os.path.join(OUT, 'claims.tsv')
claims.to_csv(path, sep='\t', index=False)
digest = hashlib.sha256(open(path, 'rb').read()).hexdigest()
manifest = dict(question='prospective_localization (one model per niche)',
                label='nodes.parquet:lopit_unified', frozen_at=time.strftime('%Y-%m-%dT%H:%M:%S'),
                parascape_version=__version__, strategy=strategy.name, strategy_hash=strategy.identity,
                niches=sorted(scores), niches_not_modelled=skipped, n_training=int(assigned.sum()),
                n_claims=int(len(claims)), n_clear_margin=int((claims.margin >= 0.2).sum()),
                evidence_columns=len(columns), claims_sha256=digest,
                held_out_record='parascape/question_registry.py: LOCALIZATION_NICHES (frozen-test AUC per niche)',
                grade_with='accuracy against the commonest-niche baseline on the genes an independent '
                           'localisation source places (new hyperLOPIT, tagging campaign or curated '
                           'per-protein papers), overall and among claims with margin >= 0.2')
json.dump(manifest, open(os.path.join(OUT, 'manifest.json'), 'w'), indent=1)
manifest

{'question': 'prospective_localization (one model per niche)', 'label': 'nodes.parquet:lopit_unified', 'frozen_at': '2026-11-08T12:59:21', 'parascape_version': '0.0.10.0', 'strategy': 'boosted trees (reference)', 'strategy_hash': '1b4ace1f7077151fe1eba96e64d52d1860b359f529b4c92f4bb2ea09e8c42901', 'niches': ['ER', 'apical_secretory', 'apicoplast', 'cytosol', 'golgi_secretory_vesicular', 'mitochondrion', 'nucleus', 'pellicle_PM_IMC', 'ribosome_RNP'], 'niches_not_modelled': ['proteasome (31)', 'cytoskeleton (9)'], 'n_training': 3827, 'n_claims': 4313, 'n_clear_margin': 1866, 'evidence_columns': 370, 'claims_sha256': '0773e04f683dd29fa88d83588ce000ecdc3d6ac07ce1931ec61a53d978102adb', 'held_out_record': 'parascape/question_registry.py: LOCALIZATION_NICHES (frozen-test AUC per niche)', 'grade_with': 'accuracy against the commonest-niche baseline on the genes an independent localisation source places (new hyperLOPIT, tagging campaign or curated per-protein papers), overall and among claims wi